# PTM Binder Workshop: Multi-design sweep around the CD3e ITAM motif

This notebook keeps the original PTM-target setup, but it samples multiple RFD3 backbones, runs LigandMPNN on every backbone, and then refolds every designed complex with RF3 so you can compare the whole candidate set side by side.


## 0. Setup

This notebook works in both Google Colab and a local Pixi setup.

- In Colab: switch to a GPU runtime and run the setup cells from the top.
- Locally: launch Jupyter from the `ptm_foundry` repo in the Pixi `dev` environment.

Recommended local setup from the repo root:

```bash
pixi install -e dev
pixi run -e dev install-workshop-kernel
pixi run -e dev workshop-notebook
```

Then open this notebook with the `PTM Workshop` kernel and run it from the top.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
GIT_URL = os.environ.get("PTM_FOUNDRY_GIT_URL", "https://github.com/magnusbauer/ptm_foundry.git")
GIT_REF = os.environ.get("PTM_FOUNDRY_GIT_REF", "workshop")

if IN_COLAB:
    REPO_DIR = Path("/content/foundry")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    REPO_DIR = next(
        (candidate for candidate in candidates if (candidate / "examples").exists() and (candidate / "models").exists()),
        cwd,
    )

if IN_COLAB and not REPO_DIR.exists():
    subprocess.check_call(
        ["git", "clone", "--branch", GIT_REF, GIT_URL, str(REPO_DIR)]
    )

SOURCE_PATHS = [
    REPO_DIR / "src",
    REPO_DIR / "models" / "rfd3" / "src",
    REPO_DIR / "models" / "mpnn" / "src",
    REPO_DIR / "models" / "rf3" / "src",
    REPO_DIR / "examples",
]
for source_path in SOURCE_PATHS:
    if source_path.exists() and str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_DIR}")
if IN_COLAB:
    print(f"Clone source:    {GIT_URL} @ {GIT_REF}")


In [ ]:
SPOOF_CIF_PATH = REPO_DIR / "examples" / "spoof_cif.py"
WORKSHOP_OUTPUT_DIR = REPO_DIR / "examples" / "workshop_outputs"

if not SPOOF_CIF_PATH.exists():
    raise FileNotFoundError(
        f"Expected workshop helper at {SPOOF_CIF_PATH}. "
        "Make sure the repo checkout includes examples/spoof_cif.py."
    )

WORKSHOP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using workshop helper: {SPOOF_CIF_PATH}")
print(f"Workshop outputs:     {WORKSHOP_OUTPUT_DIR}")


In [ ]:
%%time

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["CCD_MIRROR_PATH"] = ""
os.environ["PDB_MIRROR_PATH"] = ""

for env_name, default_value in {
    "DEBUG": "0",
    "TYPE_CHECK": "0",
    "NAN_CHECK": "1",
    "DISABLE_CUEQUIVARIANCE": "0",
}.items():
    if not os.environ.get(env_name, "").strip():
        os.environ[env_name] = default_value

CKPT_DIR = Path.home() / ".foundry" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FOUNDRY_CHECKPOINTS_DIR"] = str(CKPT_DIR)


def ensure_pip() -> None:
    if importlib.util.find_spec("pip") is None:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])


def pip_install(packages: list[str]) -> None:
    if not packages:
        return
    ensure_pip()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def start_download(url: str, dest: Path) -> tuple[subprocess.Popen, Path]:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    if tmp_dest.exists():
        tmp_dest.unlink()
    proc = subprocess.Popen(
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            "-o",
            str(tmp_dest),
            url,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return proc, tmp_dest


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

REQUIRED_PACKAGES = {
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "pandas": "pandas",
    "hydride": "hydride",
    "biotite": "biotite",
    "atomworks": "atomworks[ml]>=2.1.1",
    "lightning": "lightning>=2.5.0",
    "rootutils": "rootutils>=1.0.7,<1.1",
    "hydra": "hydra-core>=1.3.0,<1.4",
    "environs": "environs>=11.0.0,<12",
    "rich": "rich>=13.9.4",
    "jaxtyping": "jaxtyping>=0.2.17,<1",
    "beartype": "beartype>=0.18.0,<1",
    "loralib": "loralib>=0.1.1",
    "einops": "einops>=0.8.0,<1",
    "einx": "einx>=0.1.0,<1",
    "opt_einsum": "opt_einsum>=3.4.0,<4",
    "tree": "dm-tree>=0.1.6,<1",
    "zstandard": "zstandard",
    "toolz": "toolz",
    "pydantic": "pydantic>=2.8",
    "plotly": "plotly>=5,<7",
    "ipywidgets": "ipywidgets>=8,<9",
    "py3Dmol": "py3Dmol",
    "assertpy": "assertpy",
}
missing_packages = [
    package
    for module_name, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:")
    for package in missing_packages:
        print(" -", package)
    pip_install(missing_packages)
else:
    print("Python dependencies already satisfied.")

CHECKPOINTS = {
    "rfd3": {
        "url": "https://files.ipd.uw.edu/pub/rfd3/rfd3_foundry_2025_12_01_remapped.ckpt",
        "filename": "rfd3_latest.ckpt",
    },
    "ligandmpnn": {
        "url": "https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt",
        "filename": "ligandmpnn_v_32_010_25.pt",
    },
    "rf3": {
        "url": "https://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt",
        "filename": "rf3_foundry_01_24_latest_remapped.ckpt",
    },
}

download_jobs = []
for name, info in CHECKPOINTS.items():
    dest = CKPT_DIR / info["filename"]
    if dest.exists():
        print(f"{name}: already present at {dest}")
        continue
    print(f"Starting download {name} -> {dest}")
    proc, tmp_dest = start_download(info["url"], dest)
    download_jobs.append((name, dest, tmp_dest, proc))

for name, dest, tmp_dest, proc in download_jobs:
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Download failed for {name} with exit code {rc}")
    tmp_dest.replace(dest)
    print(f"{name}: downloaded to {dest}")

print("\nCheckpoint directory contents:")
for checkpoint_path in sorted(CKPT_DIR.iterdir()):
    print(" -", checkpoint_path.name)

RFD3_CKPT = CKPT_DIR / CHECKPOINTS["rfd3"]["filename"]
LIGANDMPNN_CKPT = CKPT_DIR / CHECKPOINTS["ligandmpnn"]["filename"]
RF3_CKPT = CKPT_DIR / CHECKPOINTS["rf3"]["filename"]

print("\nResolved checkpoint paths:")
print(f" - RFD3:       {RFD3_CKPT}")
print(f" - LigandMPNN: {LIGANDMPNN_CKPT}")
print(f" - RF3:        {RF3_CKPT}")


In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", module="atomworks")

EXAMPLES_DIR = REPO_DIR / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.io.utils.visualize import view
from biotite.structure import rmsd, superimpose
from lightning.fabric import seed_everything
from spoof_cif import (
    chain_summary,
    extract_chain_sequence,
    plot_residue_bond_graph,
)

from atomworks.constants import DICT_THREE_TO_ONE, UNKNOWN_AA
from atomworks.io.tools.inference import (
    build_msa_paths_by_chain_id_from_component_list,
    components_to_atom_array,
)
from atomworks.io.utils.io_utils import to_cif_file

import plotly.express as px


## 1. Define the Workshop Target

The peptide target is from an ITAM motif from CD3ε `PVPNPD(PTR)EPIRKGQ`, where `(PTR)` is phosphotyrosine. We are going to build a binder around that site, so first let's make sure the residue numbering and PTM position line up the way we expect.


In [ ]:
TARGET_SEQUENCE = "PVPNPD(PTR)EPIRKGQ"
TARGET_CHAIN_ID = "B"
BINDER_LENGTH = 100
EXAMPLE_NAME = "ptr_workshop_mult"
RFD3_DIFFUSION_BATCH_SIZE = 3
RFD3_N_BATCHES = 1
MPNN_SEQUENCES_PER_BACKBONE = 4
RF3_DIFFUSION_BATCH_SIZE = 1
WORK_DIR = Path("/content/ptm_workshop_mult") if IN_COLAB else WORKSHOP_OUTPUT_DIR / "mult"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target sequence: {TARGET_SEQUENCE}")
print(f"Target chain:    {TARGET_CHAIN_ID}")
print(f"Binder length:   {BINDER_LENGTH}")
print(f"Working dir:     {WORK_DIR}")
print(f"RFD3 sweep:      diffusion_batch_size={RFD3_DIFFUSION_BATCH_SIZE}, n_batches={RFD3_N_BATCHES}")
print(f"LigandMPNN:      {MPNN_SEQUENCES_PER_BACKBONE} sequences per backbone")
print(f"RF3 sweep:       diffusion_batch_size={RF3_DIFFUSION_BATCH_SIZE}")


## 2. Spoof the PTM CIF on the Fly

Since we need an initial bondgraph we regenerate the phosphopeptide target directly from the sequence each time so the PTM chemistry is explicit and reproducible.

To keep the flow close to the original notebook, we first build the cifutils-style component dictionary from the sequence and then pass that dictionary into `spoof_cif_from_dictionary(...)`.

The peptide is fixed in sequence because chain `B` comes from this input CIF and later `LigandMPNN` only designs chain `A`. The peptide can still change in structure because `select_fixed_atoms` stays `false`, so RFD3 is allowed to move the coordinates even while the residue identities and PTM chemistry stay fixed.


In [ ]:
spoof_input = { 'name': 'ptr_workshop',
                'components': [{'seq': 'PVPNPD(PTR)EPIRKGQ', 'chain_id': 'B'}]}

In [ ]:
atom_array, component_list = components_to_atom_array(
    spoof_input["components"],
    return_components=True,
    bonds=spoof_input.get("bonds"),
)

cif_path = WORK_DIR / f"{spoof_input['name']}.cif"
RFD3_JSON_PATH = WORK_DIR / f"{spoof_input['name']}.json"

save_path = Path(
    to_cif_file(
        atom_array,
        cif_path,
        file_type="cif",
    )
)

print(f"Spoofed CIF:   {cif_path}")

In [ ]:
atom_array

## 3. Inspect the Bond Graph

Before running design, it helps to look at the chemistry from a few angles. The next cells show:

- a 2D atom graph with the bond types written on each edge


In [ ]:
bond_array = atom_array.bonds.as_array()

In [ ]:
pairs = bond_array[:, :2]                               # atom index pairs
bond_types = bond_array[:, 2]                           # bond order/type

bond_mat = atom_array.bonds.bond_type_matrix()          # bond types, -1 if no bond

In [ ]:
display_mat = ~bond_mat

atom_labels = np.array([
    f"{i}: {atom_array.chain_id[i]}{int(atom_array.res_id[i])} "
    f"{atom_array.res_name[i]}:{atom_array.atom_name[i]}"
    for i in range(atom_array.array_length())
], dtype=object)

bond_type_mat = atom_array.bonds.bond_type_matrix()
bond_type_map = {
    -1: "no bond",
    0: "any",
    1: "single",
    2: "double",
    3: "triple",
    4: "quadruple",
    5: "aromatic single",
    6: "aromatic double",
    7: "aromatic triple",
    8: "coordination",
    9: "aromatic",
}
bond_type_labels = np.vectorize(lambda x: bond_type_map.get(int(x), f"unknown ({x})"))(bond_type_mat)

fig = px.imshow(
    display_mat,
    x=atom_labels,
    y=atom_labels,
    origin="lower",
)

fig.update_traces(
    text=bond_type_labels,
    hovertemplate=(
        "atom i: %{y}<br>"
        "atom j: %{x}<br>"
        "bond type: %{text}<br>"
        "shown value: %{z}<extra></extra>"
    )
)

fig.update_layout(
    width=800,
    height=800,
)

fig.update_xaxes(showticklabels=False)

fig.show()

## 4. Exercise: Write the RFD3 JSON

Edit the `json_data` cell above to finish the selector fields for a `100` residue binder against the phosphorylated peptide.

Use the bond tables and graphs above to decide what goes into:
- `select_hotspots`: atoms that should anchor interface orientation around the PTM.
- `select_buried`: atoms you want packed against the binder.
- `select_hbond_acceptor`: atoms that should accept H-bonds from the binder.

Keep the rest of the JSON scaffold unchanged.

About `OH`: in `PTR`, `OH` is the tyrosine side-chain oxygen that links the aromatic ring to the phosphate. It is part of the phosphotyrosine residue, not a separate free hydroxyl group floating off the PTM.


In [ ]:
json_data = {
    EXAMPLE_NAME: {
        "input": str(cif_path),
        "contig": f"100-100,/0,B1-14",
        "dialect": 2,
        "infer_ori_strategy": "hotspots",
        "redesign_motif_sidechains": False,
        "select_fixed_atoms": False,
        "select_hotspots": {
                  "B7": "P,O1P,O2P,O3P,OH"# Fill this in.
        },
        "select_buried": {
            # Fill this in.
        },
        "select_hbond_acceptor": {
            # Fill this in.
        },
    }
}


In [ ]:
print(json.dumps(json_data, indent=2))

with open(RFD3_JSON_PATH, "w") as f:
    json.dump(json_data, f, indent=2)


## 5. Generate Multiple Binder Backbones with RFD3

Instead of keeping only one binder backbone, this notebook samples several in one RFD3 pass. Every backbone is carried forward through LigandMPNN and RF3 so the final score table covers the full sweep.


In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if cuda_available:
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA device detected. The RFD3 / LigandMPNN / RF3 cells are workshop GPU steps and can be extremely slow on CPU.")


In [ ]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

seed_everything(7)

rfd3_config = RFD3InferenceConfig(
    ckpt_path=str(RFD3_CKPT),
    diffusion_batch_size=RFD3_DIFFUSION_BATCH_SIZE,
)
rfd3_engine = RFD3InferenceEngine(**rfd3_config)
rfd3_outputs = rfd3_engine.run(
    inputs=str(RFD3_JSON_PATH),
    out_dir=None,
    n_batches=RFD3_N_BATCHES,
)

rfd3_designs = []
for example_id, per_example_outputs in rfd3_outputs.items():
    for model_index, output in enumerate(per_example_outputs):
        design_name = f"{example_id}_rfd3_m{model_index}"
        binder_token_count = len(
            np.unique(output.atom_array[output.atom_array.chain_id == "A"].res_id)
        )
        rfd3_designs.append(
            {
                "design_name": design_name,
                "rfd3_example_id": example_id,
                "rfd3_model_index": model_index,
                "binder_token_count": binder_token_count,
                "atom_array": output.atom_array,
                "rfd3_output": output,
            }
        )

rfd3_summary = pd.DataFrame(
    [
        {
            "design_name": record["design_name"],
            "rfd3_example_id": record["rfd3_example_id"],
            "rfd3_model_index": record["rfd3_model_index"],
            "binder_token_count": record["binder_token_count"],
        }
        for record in rfd3_designs
    ]
)
display(rfd3_summary)
print(f"Generated {len(rfd3_designs)} RFD3 backbone complexes.")


### Studio-style RFD3 Browser

This viewer is laid out like a compact Studio panel: the selected RFD3 structure is on the left, and a pairwise binder-backbone RMSD heatmap plus the selected row metadata are on the right. Use the buttons to move back and forth through the generated backbones.


In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import clear_output
import biotite.structure as struc
import py3Dmol
from atomworks.io.utils.io_utils import to_cif_string

BROWSER_PTR_RESIDUE_ID = 7
PTR_SIDECHAIN_ATOMS = [
    "CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ", "OH",
    "P", "O1P", "O2P", "O3P",
]

def make_browser_structure_view(primary, secondary=None, width=520, height=430, ptr_chain="B", ptr_resi=BROWSER_PTR_RESIDUE_ID):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(primary, include_entity_poly=False, _allow_ambiguous_bond_annotations=True),
        "mmcif",
    )
    if secondary is not None:
        viewer.addModel(
            to_cif_string(secondary, include_entity_poly=False, _allow_ambiguous_bond_annotations=True),
            "mmcif",
        )
        viewer.setStyle({"model": 0}, {"cartoon": {"color": "#64748b", "opacity": 0.35}})
        viewer.setStyle({"model": 1}, {"cartoon": {"color": "#dc2626", "opacity": 0.85}})
    else:
        viewer.setStyle({"model": 0, "chain": "A"}, {"cartoon": {"color": "#2563eb", "opacity": 0.95}})
        viewer.setStyle({"model": 0, "chain": ptr_chain}, {"cartoon": {"color": "#94a3b8", "opacity": 0.85}})
    if ptr_resi is not None:
        viewer.setStyle(
            {"model": 0, "chain": ptr_chain, "resi": int(ptr_resi), "resn": "PTR", "atom": PTR_SIDECHAIN_ATOMS},
            {"stick": {"color": "#f59e0b", "radius": 0.18}},
        )
        if secondary is not None:
            viewer.setStyle(
                {"model": 1, "chain": ptr_chain, "resi": int(ptr_resi), "resn": "PTR", "atom": PTR_SIDECHAIN_ATOMS},
                {"stick": {"color": "#dc2626", "radius": 0.22}},
            )
        viewer.zoomTo({"chain": ptr_chain, "resi": int(ptr_resi), "resn": "PTR"})
    else:
        viewer.zoomTo()
    return viewer

def paired_backbone_indices(reference, mobile, chain_id="A"):
    ref_mask = (reference.chain_id == chain_id) & np.isin(reference.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    mobile_mask = (mobile.chain_id == chain_id) & np.isin(mobile.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    ref_indices = np.flatnonzero(ref_mask)
    mobile_indices = np.flatnonzero(mobile_mask)
    ref_lookup = {(reference.chain_id[i], int(reference.res_id[i]), reference.atom_name[i]): i for i in ref_indices}
    mobile_lookup = {(mobile.chain_id[i], int(mobile.res_id[i]), mobile.atom_name[i]): i for i in mobile_indices}
    common_keys = [key for key in ref_lookup if key in mobile_lookup]
    if not common_keys:
        raise ValueError("No common binder backbone atoms found between structures")
    ref_paired = np.array([ref_lookup[key] for key in common_keys], dtype=int)
    mobile_paired = np.array([mobile_lookup[key] for key in common_keys], dtype=int)
    return ref_paired, mobile_paired

def binder_backbone_rmsd(reference, mobile, chain_id="A"):
    ref_idx, mobile_idx = paired_backbone_indices(reference, mobile, chain_id=chain_id)
    _, transform = struc.superimpose(reference[ref_idx], mobile[mobile_idx])
    mobile_fit = transform.apply(mobile)
    return float(struc.rmsd(reference[ref_idx], mobile_fit[mobile_idx]))

def extract_min_interface_pae(summary_confidences):
    matrix = summary_confidences.get("chain_pair_pae_min") or []
    values = [
        float(value)
        for i, row in enumerate(matrix)
        for j, value in enumerate(row)
        if i != j and value is not None
    ]
    return float(min(values)) if values else np.nan


In [ ]:
rfd3_browser_df = rfd3_summary.copy()
rfd3_browser_df["atom_count"] = [int(len(record["atom_array"])) for record in rfd3_designs]

rfd3_pairwise_rmsd = np.zeros((len(rfd3_designs), len(rfd3_designs)), dtype=float)
for i, ref_record in enumerate(rfd3_designs):
    for j in range(i + 1, len(rfd3_designs)):
        rmsd_value = binder_backbone_rmsd(ref_record["atom_array"], rfd3_designs[j]["atom_array"])
        rfd3_pairwise_rmsd[i, j] = rmsd_value
        rfd3_pairwise_rmsd[j, i] = rmsd_value

rfd3_labels = rfd3_browser_df["design_name"].tolist()
rfd3_index = {"value": 0}
rfd3_prev_button = widgets.Button(description="Previous")
rfd3_next_button = widgets.Button(description="Next")
rfd3_status_html = widgets.HTML()
rfd3_left_output = widgets.Output(layout=widgets.Layout(width="52%"))
rfd3_right_output = widgets.Output(layout=widgets.Layout(width="48%"))

def make_rfd3_heatmap(selected_idx):
    fig = go.Figure(
        data=go.Heatmap(
            z=rfd3_pairwise_rmsd,
            x=rfd3_labels,
            y=rfd3_labels,
            colorscale="Blues",
            colorbar_title="RMSD (A)",
            hovertemplate="ref=%{y}<br>mobile=%{x}<br>RMSD=%{z:.2f} A<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[rfd3_labels[selected_idx]],
            y=[rfd3_labels[selected_idx]],
            mode="markers",
            marker={"size": 14, "color": "#dc2626", "symbol": "x"},
            showlegend=False,
            hoverinfo="skip",
        )
    )
    fig.update_layout(
        title="Binder-backbone RMSD across RFD3 candidates",
        height=380,
        margin=dict(l=0, r=0, t=45, b=0),
    )
    return fig

def render_rfd3_candidate():
    idx = rfd3_index["value"]
    record = rfd3_designs[idx]
    row = rfd3_browser_df.iloc[idx]
    rfd3_status_html.value = (
        f"<b>{idx + 1}/{len(rfd3_designs)}</b> &nbsp; <code>{row['design_name']}</code> &nbsp; "
        f"binder residues={row['binder_token_count']} &nbsp; atoms={row['atom_count']}"
    )
    rfd3_prev_button.disabled = idx == 0
    rfd3_next_button.disabled = idx == len(rfd3_designs) - 1
    with rfd3_left_output:
        clear_output(wait=True)
        display(make_browser_structure_view(record["atom_array"]))
    with rfd3_right_output:
        clear_output(wait=True)
        display(make_rfd3_heatmap(idx))
        display(pd.DataFrame([row]).round(3))

def step_rfd3(delta):
    rfd3_index["value"] = min(max(rfd3_index["value"] + delta, 0), len(rfd3_designs) - 1)
    render_rfd3_candidate()

rfd3_prev_button.on_click(lambda _: step_rfd3(-1))
rfd3_next_button.on_click(lambda _: step_rfd3(1))

display(
    widgets.VBox(
        [
            widgets.HBox([rfd3_prev_button, rfd3_next_button, rfd3_status_html]),
            widgets.HBox([rfd3_left_output, rfd3_right_output]),
        ]
    )
)
render_rfd3_candidate()


## 6. Design Binder Sequences for Every Backbone

LigandMPNN now runs on every RFD3 backbone instead of just the first one. Each backbone gets its own small sequence sweep so the final comparison keeps both backbone diversity and sequence diversity.


In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

mpnn_engine = MPNNInferenceEngine(
    model_type="ligand_mpnn",
    checkpoint_path=str(LIGANDMPNN_CKPT),
    is_legacy_weights=True,
    out_directory=None,
    write_structures=False,
    write_fasta=False,
)

mpnn_input_dicts = [
    {
        "name": record["design_name"],
        "batch_size": MPNN_SEQUENCES_PER_BACKBONE,
        "remove_waters": True,
        "designed_chains": ["A"],
    }
    for record in rfd3_designs
]

mpnn_outputs = mpnn_engine.run(
    input_dicts=mpnn_input_dicts,
    atom_arrays=[record["atom_array"] for record in rfd3_designs],
)

rfd3_by_name = {record["design_name"]: record for record in rfd3_designs}
mpnn_designs = []
for output in mpnn_outputs:
    parent_name = output.input_dict["name"]
    design_idx = int(output.output_dict["design_idx"])
    parent_record = rfd3_by_name[parent_name]
    mpnn_name = f"{parent_name}_mpnn_d{design_idx}"
    interface_recovery = output.output_dict["ligand_interface_sequence_recovery"]
    mpnn_designs.append(
        {
            "design_name": mpnn_name,
            "parent_name": parent_name,
            "rfd3_example_id": parent_record["rfd3_example_id"],
            "rfd3_model_index": parent_record["rfd3_model_index"],
            "rfd3_complex": parent_record["atom_array"],
            "mpnn_design_idx": design_idx,
            "mpnn_output": output,
            "reference_complex": output.atom_array,
            "binder_sequence": extract_chain_sequence(output.atom_array, "A"),
            "target_sequence": extract_chain_sequence(output.atom_array, TARGET_CHAIN_ID),
            "sequence_recovery": float(output.output_dict["sequence_recovery"]),
            "ligand_interface_sequence_recovery": (
                float(interface_recovery) if interface_recovery is not None else np.nan
            ),
        }
    )

mpnn_summary = pd.DataFrame(
    [
        {
            "design_name": record["design_name"],
            "parent_name": record["parent_name"],
            "rfd3_model_index": record["rfd3_model_index"],
            "mpnn_design_idx": record["mpnn_design_idx"],
            "binder_sequence": record["binder_sequence"],
            "sequence_recovery": record["sequence_recovery"],
            "ligand_interface_sequence_recovery": record["ligand_interface_sequence_recovery"],
        }
        for record in mpnn_designs
    ]
)
display(mpnn_summary)
print(f"Generated {len(mpnn_designs)} LigandMPNN sequence designs across {len(rfd3_designs)} backbones.")


## 7. Refold Every Designed Complex with RF3

RF3 now refolds the whole LigandMPNN candidate set so we can score every binder-target complex instead of manually picking one design up front.


In [ ]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

rf3_engine = RF3InferenceEngine(
    ckpt_path=str(RF3_CKPT),
    diffusion_batch_size=RF3_DIFFUSION_BATCH_SIZE,
    verbose=False,
)

rf3_inputs = [
    InferenceInput.from_atom_array(
        record["reference_complex"],
        example_id=record["design_name"],
    )
    for record in mpnn_designs
]

rf3_outputs = rf3_engine.run(
    inputs=rf3_inputs,
    annotate_b_factor_with_plddt=True,
)

rf3_designs = []
for record in mpnn_designs:
    example_id = record["design_name"]
    per_example_outputs = rf3_outputs[example_id]
    for sample_idx, rf3_output in enumerate(per_example_outputs):
        rf3_name = example_id if len(per_example_outputs) == 1 else f"{example_id}_rf3_s{sample_idx}"
        rf3_designs.append(
            {
                **record,
                "rf3_name": rf3_name,
                "rf3_sample_idx": sample_idx,
                "rf3_output": rf3_output,
                "summary_confidences": rf3_output.summary_confidences,
            }
        )

rf3_summary = pd.DataFrame(
    [
        {
            "rf3_name": record["rf3_name"],
            "design_name": record["design_name"],
            "overall_plddt": float(record["summary_confidences"].get("overall_plddt", np.nan)),
            "overall_pae": float(record["summary_confidences"].get("overall_pae", np.nan)),
            "ptm": (
                float(record["summary_confidences"].get("ptm"))
                if record["summary_confidences"].get("ptm") is not None
                else np.nan
            ),
            "iptm": (
                float(record["summary_confidences"].get("iptm"))
                if record["summary_confidences"].get("iptm") is not None
                else np.nan
            ),
            "ranking_score": (
                float(record["summary_confidences"].get("ranking_score"))
                if record["summary_confidences"].get("ranking_score") is not None
                else np.nan
            ),
        }
        for record in rf3_designs
    ]
)
display(rf3_summary)
print(f"Ran RF3 on {len(rf3_designs)} designed complexes.")


### Studio-style RF3 Browser

This viewer shows the RF3 refold directly after the refold step. The left pane overlays the LigandMPNN design with the RF3 result, and the right pane shows a confidence scatter with the selected design highlighted plus its row-level RF3 metrics.


In [ ]:
rf3_browser_df = rf3_summary.copy()
rf3_browser_df["overall_plddt_pct"] = rf3_browser_df["overall_plddt"] * 100.0
rf3_browser_df["min_pae"] = [extract_min_interface_pae(record["summary_confidences"]) for record in rf3_designs]

rf3_index = {"value": 0}
rf3_prev_button = widgets.Button(description="Previous")
rf3_next_button = widgets.Button(description="Next")
rf3_status_html = widgets.HTML()
rf3_note_html = widgets.HTML("<span style='color:#475569'>Gray = LigandMPNN design, red = RF3 refold.</span>")
rf3_left_output = widgets.Output(layout=widgets.Layout(width="52%"))
rf3_right_output = widgets.Output(layout=widgets.Layout(width="48%"))

def make_rf3_confidence_scatter(selected_idx):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=rf3_browser_df["min_pae"],
            y=rf3_browser_df["overall_plddt_pct"],
            mode="markers",
            marker={"size": 10, "color": "#60a5fa", "line": {"width": 1, "color": "white"}},
            text=rf3_browser_df["rf3_name"],
            customdata=np.stack([rf3_browser_df["overall_pae"], rf3_browser_df["ranking_score"]], axis=1),
            hovertemplate=(
                "%{text}<br>min_pae=%{x:.2f} A<br>pLDDT=%{y:.1f}%<br>overall_pae=%{customdata[0]:.2f} A"
                "<br>ranking_score=%{customdata[1]:.3f}<extra></extra>"
            ),
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[rf3_browser_df.iloc[selected_idx]["min_pae"]],
            y=[rf3_browser_df.iloc[selected_idx]["overall_plddt_pct"]],
            mode="markers",
            marker={"size": 16, "color": "#dc2626", "symbol": "diamond-open", "line": {"width": 2}},
            hoverinfo="skip",
            showlegend=False,
        )
    )
    fig.update_layout(
        title="RF3 min-PAE vs pLDDT",
        xaxis_title="Minimum interface PAE (A)",
        yaxis_title="Overall pLDDT (%)",
        height=380,
        margin=dict(l=0, r=0, t=45, b=0),
    )
    return fig

def render_rf3_candidate():
    idx = rf3_index["value"]
    record = rf3_designs[idx]
    row = rf3_browser_df.iloc[idx]
    rf3_status_html.value = (
        f"<b>{idx + 1}/{len(rf3_designs)}</b> &nbsp; <code>{row['rf3_name']}</code> &nbsp; "
        f"pLDDT={row['overall_plddt_pct']:.1f}% &nbsp; min_PAE={row['min_pae']:.2f} A"
    )
    rf3_prev_button.disabled = idx == 0
    rf3_next_button.disabled = idx == len(rf3_designs) - 1
    with rf3_left_output:
        clear_output(wait=True)
        display(make_browser_structure_view(record["reference_complex"], record["rf3_output"].atom_array))
    with rf3_right_output:
        clear_output(wait=True)
        display(make_rf3_confidence_scatter(idx))
        display(pd.DataFrame([row]).round(3))

def step_rf3(delta):
    rf3_index["value"] = min(max(rf3_index["value"] + delta, 0), len(rf3_designs) - 1)
    render_rf3_candidate()

rf3_prev_button.on_click(lambda _: step_rf3(-1))
rf3_next_button.on_click(lambda _: step_rf3(1))

display(
    widgets.VBox(
        [
            widgets.HBox([rf3_prev_button, rf3_next_button, rf3_status_html]),
            rf3_note_html,
            widgets.HBox([rf3_left_output, rf3_right_output]),
        ]
    )
)
render_rf3_candidate()


## 8. Score Every Candidate

The helper functions below are the same notebook checks as the single-design workshop, but now they run across the full candidate set so the last table has one row per refolded complex.


In [ ]:
import biotite.structure as struc
import hydride
import py3Dmol
from atomworks.io.utils.io_utils import to_cif_string

HBOND_CUTOFF_DIST = 2.5
HBOND_CUTOFF_ANGLE = 120.0
HBOND_PH = 7.0
HBOND_REQUIRE_CROSS_RESIDUE = True

SASA_PROBE_RADIUS = 1.4
SASA_VDW_RADII = "Single"
SASA_POINT_NUMBER = 1000
SASA_POINT_DISTR = "Fibonacci"

FINAL_FILTER_MAX_PEPTIDE_CA_RMSD = 1.5
FINAL_FILTER_MIN_PO4_BURIAL = 0.35
FINAL_FILTER_MIN_HBONDS = 2

PHOSPHATE_ATOMS = ("P", "O1P", "O2P", "O3P")

ptm_residue_id = 7


In [ ]:
def paired_common_indices(reference, mobile, ref_mask, mobile_mask):
    ref_indices = np.flatnonzero(ref_mask)
    mobile_indices = np.flatnonzero(mobile_mask)

    ref_lookup = {
        (reference.chain_id[i], int(reference.res_id[i]), reference.atom_name[i]): i
        for i in ref_indices
    }
    mobile_lookup = {
        (mobile.chain_id[i], int(mobile.res_id[i]), mobile.atom_name[i]): i
        for i in mobile_indices
    }

    common_keys = [key for key in ref_lookup if key in mobile_lookup]
    if not common_keys:
        raise ValueError("No common atoms found between the reference and mobile selections")

    ref_paired = np.array([ref_lookup[key] for key in common_keys], dtype=int)
    mobile_paired = np.array([mobile_lookup[key] for key in common_keys], dtype=int)
    return ref_paired, mobile_paired


def align_mobile_on_binder_backbone(reference, mobile, ref_mask, mobile_mask):
    ref_idx, mobile_idx = paired_common_indices(reference, mobile, ref_mask, mobile_mask)
    _, transform = struc.superimpose(reference[ref_idx], mobile[mobile_idx])
    mobile_aligned = transform.apply(mobile)
    alignment_rmsd = float(struc.rmsd(reference[ref_idx], mobile_aligned[mobile_idx]))
    return mobile_aligned, transform, alignment_rmsd


def rmsd_for_masks(reference, mobile, ref_mask, mobile_mask, allow_mismatch=False):
    if allow_mismatch:
        ref_idx, mobile_idx = paired_common_indices(reference, mobile, ref_mask, mobile_mask)
    else:
        ref_idx = np.flatnonzero(ref_mask)
        mobile_idx = np.flatnonzero(mobile_mask)
        if len(ref_idx) != len(mobile_idx):
            raise ValueError(
                f"Selection size mismatch: reference={len(ref_idx)} mobile={len(mobile_idx)}"
            )

    if len(ref_idx) == 0:
        raise ValueError("Selection matched no atoms")

    metric_rmsd = float(struc.rmsd(reference[ref_idx], mobile[mobile_idx]))
    return metric_rmsd, len(ref_idx)


def atom_triplet_label(atom_array, atom_index):
    return (
        f"{atom_array.chain_id[atom_index]}:"
        f"{atom_array.res_name[atom_index]}{atom_array.res_id[atom_index]}:"
        f"{atom_array.atom_name[atom_index]}"
    )


def same_residue(atom_array, atom_i, atom_j):
    return (
        atom_array.chain_id[atom_i] == atom_array.chain_id[atom_j]
        and atom_array.res_id[atom_i] == atom_array.res_id[atom_j]
    )


def format_hbond_connection(atom_array, donor_idx, hydrogen_idx, acceptor_idx):
    donor = atom_triplet_label(atom_array, donor_idx)
    hydrogen = atom_array.atom_name[hydrogen_idx]
    acceptor = atom_triplet_label(atom_array, acceptor_idx)
    return f"{donor} -- {hydrogen} --> {acceptor}"


def prepare_atom_array_for_hbonds(atom_array, ph=HBOND_PH):
    prepared = atom_array.copy()
    if "H" in prepared.element:
        prepared = prepared[prepared.element != "H"]

    for category in list(prepared.get_annotation_categories()):
        annotation = prepared.get_annotation(category)
        if getattr(annotation, "ndim", 1) != 1:
            prepared.del_annotation(category)

    prepared.bonds = struc.connect_via_residue_names(prepared)
    prepared.charge = hydride.estimate_amino_acid_charges(prepared, ph=ph)
    prepared_with_h, _ = hydride.add_hydrogen(prepared)
    prepared_with_h.coord = hydride.relax_hydrogen(prepared_with_h)
    return prepared_with_h


def compute_phosphosite_hbond_metrics(atom_array, chain_id, residue_id, res_name="PTR"):
    prepared = prepare_atom_array_for_hbonds(atom_array, ph=HBOND_PH)
    phosphosite_mask = (
        (prepared.chain_id == chain_id)
        & (prepared.res_id == residue_id)
        & (prepared.res_name == res_name)
    )
    if not phosphosite_mask.any():
        raise ValueError(f"No atoms found for {chain_id}:{res_name}{residue_id}")

    hbond_result = struc.hbond(
        prepared,
        selection1_type="both",
        cutoff_dist=HBOND_CUTOFF_DIST,
        cutoff_angle=HBOND_CUTOFF_ANGLE,
    )
    triplets = hbond_result[0] if isinstance(hbond_result, tuple) else hbond_result

    all_connections = []
    phosphosite_connections = []
    phosphosite_records = []
    for donor_idx, hydrogen_idx, acceptor_idx in triplets:
        if HBOND_REQUIRE_CROSS_RESIDUE and same_residue(prepared, donor_idx, acceptor_idx):
            continue

        connection = format_hbond_connection(
            prepared,
            donor_idx=donor_idx,
            hydrogen_idx=hydrogen_idx,
            acceptor_idx=acceptor_idx,
        )
        all_connections.append(connection)

        donor_match = bool(phosphosite_mask[donor_idx])
        acceptor_match = bool(phosphosite_mask[acceptor_idx])
        if donor_match or acceptor_match:
            phosphosite_connections.append(connection)
            phosphosite_records.append(
                {
                    "donor_idx": int(donor_idx),
                    "hydrogen_idx": int(hydrogen_idx),
                    "acceptor_idx": int(acceptor_idx),
                    "donor_label": atom_triplet_label(prepared, donor_idx),
                    "hydrogen_label": atom_triplet_label(prepared, hydrogen_idx),
                    "acceptor_label": atom_triplet_label(prepared, acceptor_idx),
                    "donor_acceptor_distance": float(
                        np.linalg.norm(prepared.coord[donor_idx] - prepared.coord[acceptor_idx])
                    ),
                    "hydrogen_acceptor_distance": float(
                        np.linalg.norm(prepared.coord[hydrogen_idx] - prepared.coord[acceptor_idx])
                    ),
                }
            )

    return {
        "prepared_structure": prepared,
        "total_hbonds": float(len(all_connections)),
        "phosphosite_hbonds": float(len(phosphosite_connections)),
        "connections": all_connections,
        "phosphosite_connections": phosphosite_connections,
        "phosphosite_records": phosphosite_records,
    }


def make_structure_overlay_view(
    reference,
    mobile,
    zoom_to_selection=None,
    width=700,
    height=500,
    ptr_chain="B",
    ptr_resi=None,
    reference_color="#64748b",
    mobile_color="#dc2626",
):
    ptr_sidechain_atoms = [
        "CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ", "OH", "P", "O1P", "O2P", "O3P"
    ]

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            reference,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.addModel(
        to_cif_string(
            mobile,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {"model": 0},
        {"cartoon": {"color": reference_color, "opacity": 0.35}},
    )
    viewer.setStyle(
        {"model": 1},
        {"cartoon": {"color": mobile_color, "opacity": 0.85}},
    )

    ptr_selection = None
    if ptr_resi is not None:
        ptr_selection = {
            "chain": ptr_chain,
            "resi": int(ptr_resi),
            "resn": "PTR",
            "atom": ptr_sidechain_atoms,
        }
        viewer.setStyle(
            {"model": 0, **ptr_selection},
            {"stick": {"color": reference_color, "radius": 0.16}},
        )
        viewer.setStyle(
            {"model": 1, **ptr_selection},
            {"stick": {"color": mobile_color, "radius": 0.22}},
        )

    if zoom_to_selection is not None:
        viewer.setStyle(
            {"model": 0, **zoom_to_selection},
            {"stick": {"color": reference_color, "radius": 0.18}, "sphere": {"color": reference_color, "scale": 0.18}},
        )
        viewer.setStyle(
            {"model": 1, **zoom_to_selection},
            {"stick": {"color": mobile_color, "radius": 0.22}, "sphere": {"color": mobile_color, "scale": 0.18}},
        )
        viewer.zoomTo(zoom_to_selection)
    elif ptr_selection is not None:
        viewer.zoomTo(ptr_selection)
    else:
        viewer.zoomTo()
    return viewer


def make_hbond_view(atom_array, hbond_records, chain_id, residue_id, width=700, height=500):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            atom_array,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {},
        {"cartoon": {"color": "#94a3b8", "opacity": 0.45}, "stick": {"colorscheme": "lightgrayCarbon", "radius": 0.1}},
    )
    viewer.setStyle(
        {"chain": chain_id, "resi": int(residue_id)},
        {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}, "sphere": {"scale": 0.22}},
    )
    for record in hbond_records:
        donor = atom_array.coord[record["donor_idx"]]
        acceptor = atom_array.coord[record["acceptor_idx"]]
        viewer.addSphere(
            {
                "center": {"x": float(donor[0]), "y": float(donor[1]), "z": float(donor[2])},
                "radius": 0.35,
                "color": "#2563eb",
                "opacity": 0.9,
            }
        )
        viewer.addSphere(
            {
                "center": {"x": float(acceptor[0]), "y": float(acceptor[1]), "z": float(acceptor[2])},
                "radius": 0.35,
                "color": "#f59e0b",
                "opacity": 0.9,
            }
        )
        viewer.addLine(
            {
                "start": {"x": float(donor[0]), "y": float(donor[1]), "z": float(donor[2])},
                "end": {"x": float(acceptor[0]), "y": float(acceptor[1]), "z": float(acceptor[2])},
                "color": "#f59e0b",
                "dashed": True,
            }
        )
    viewer.zoomTo({"chain": chain_id, "resi": int(residue_id)})
    return viewer


def project_points_to_plane(coords):
    centered = coords - coords.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    basis = vh[:2]
    return centered @ basis.T


def plot_hbond_contact_map(atom_array, hbond_records, chain_id, residue_id, phosphate_atom_names, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))

    if not hbond_records:
        ax.text(0.5, 0.5, "No phosphosite H-bonds detected", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return ax

    residue_mask = (atom_array.chain_id == chain_id) & (atom_array.res_id == residue_id)
    phosphate_mask = residue_mask & np.isin(atom_array.atom_name, phosphate_atom_names)

    residue_indices = set(np.flatnonzero(residue_mask))
    phosphate_indices = set(np.flatnonzero(phosphate_mask))
    donor_indices = {int(record["donor_idx"]) for record in hbond_records}
    acceptor_indices = {int(record["acceptor_idx"]) for record in hbond_records}
    selected_indices = np.array(sorted(residue_indices | donor_indices | acceptor_indices), dtype=int)

    projected = project_points_to_plane(atom_array.coord[selected_indices])
    point_by_index = {atom_idx: projected[pos] for pos, atom_idx in enumerate(selected_indices)}

    for record in hbond_records:
        donor_xy = point_by_index[int(record["donor_idx"])]
        acceptor_xy = point_by_index[int(record["acceptor_idx"])]
        ax.plot(
            [donor_xy[0], acceptor_xy[0]],
            [donor_xy[1], acceptor_xy[1]],
            linestyle="--",
            linewidth=1.8,
            color="#f59e0b",
            alpha=0.9,
        )

    residue_coords = np.array([point_by_index[idx] for idx in sorted(residue_indices)])
    phosphate_coords = np.array([point_by_index[idx] for idx in sorted(phosphate_indices)])
    donor_coords = np.array([point_by_index[idx] for idx in sorted(donor_indices)])
    acceptor_coords = np.array([point_by_index[idx] for idx in sorted(acceptor_indices)])

    if len(residue_coords):
        ax.scatter(residue_coords[:, 0], residue_coords[:, 1], s=110, color="#fcd34d", edgecolors="black", linewidths=0.4, zorder=2)
    if len(phosphate_coords):
        ax.scatter(phosphate_coords[:, 0], phosphate_coords[:, 1], s=170, color="#ef4444", edgecolors="black", linewidths=0.6, zorder=3)
    if len(donor_coords):
        ax.scatter(donor_coords[:, 0], donor_coords[:, 1], s=120, color="#2563eb", edgecolors="black", linewidths=0.4, zorder=4)
    if len(acceptor_coords):
        ax.scatter(acceptor_coords[:, 0], acceptor_coords[:, 1], s=120, color="#f59e0b", edgecolors="black", linewidths=0.4, zorder=4)

    label_indices = sorted(phosphate_indices | donor_indices | acceptor_indices)
    for atom_idx in label_indices:
        x, y = point_by_index[atom_idx]
        ax.text(
            x + 0.15,
            y + 0.15,
            f"{atom_array.res_name[atom_idx]}:{atom_array.atom_name[atom_idx]}",
            fontsize=8,
            zorder=5,
        )

    ax.set_title(f"Phosphosite H-bond sketch around {chain_id}{residue_id}")
    ax.set_aspect("equal")
    ax.set_axis_off()
    return ax


def make_sasa_view(atom_array, chain_id, residue_id, phosphate_atom_names, width=700, height=500):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            atom_array,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {},
        {"cartoon": {"color": "#94a3b8", "opacity": 0.3}, "stick": {"colorscheme": "lightgrayCarbon", "radius": 0.08}},
    )
    viewer.addSurface(
        py3Dmol.VDW,
        {"opacity": 0.35, "color": "white"},
        {"chain": chain_id, "resi": int(residue_id)},
    )
    viewer.setStyle(
        {"chain": chain_id, "resi": int(residue_id)},
        {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}},
    )

    phosphate_mask = (
        (atom_array.chain_id == chain_id)
        & (atom_array.res_id == residue_id)
        & np.isin(atom_array.atom_name, phosphate_atom_names)
    )
    for coord in atom_array.coord[phosphate_mask]:
        viewer.addSphere(
            {
                "center": {"x": float(coord[0]), "y": float(coord[1]), "z": float(coord[2])},
                "radius": 0.5,
                "color": "#ef4444",
                "opacity": 0.85,
            }
        )

    viewer.zoomTo({"chain": chain_id, "resi": int(residue_id)})
    return viewer


def compute_selection_sasa_metrics(atom_array, mask):
    if not np.any(mask):
        raise ValueError("Selection matched no atoms for SASA calculation")

    sasa_kwargs = {
        "probe_radius": SASA_PROBE_RADIUS,
        "vdw_radii": SASA_VDW_RADII,
        "point_number": SASA_POINT_NUMBER,
        "point_distr": SASA_POINT_DISTR,
    }
    full_complex_sasa = struc.sasa(atom_array, **sasa_kwargs)
    isolated_subset = atom_array[mask]
    isolated_sasa = struc.sasa(isolated_subset, **sasa_kwargs)

    total_iso = float(np.nansum(isolated_sasa))
    total_complex = float(np.nansum(full_complex_sasa[mask]))
    buried = float(total_iso - total_complex)
    fraction_buried = float(buried / total_iso) if total_iso > 0 else float("nan")
    return {
        "total_iso": total_iso,
        "total_complex": total_complex,
        "buried": buried,
        "fraction_buried": fraction_buried,
    }


def compute_selection_sasa_breakdown(atom_array, mask):
    if not np.any(mask):
        raise ValueError("Selection matched no atoms for SASA calculation")

    sasa_kwargs = {
        "probe_radius": SASA_PROBE_RADIUS,
        "vdw_radii": SASA_VDW_RADII,
        "point_number": SASA_POINT_NUMBER,
        "point_distr": SASA_POINT_DISTR,
    }
    full_complex_sasa = struc.sasa(atom_array, **sasa_kwargs)
    isolated_subset = atom_array[mask]
    isolated_sasa = struc.sasa(isolated_subset, **sasa_kwargs)

    rows = []
    selected_indices = np.flatnonzero(mask)
    for local_idx, atom_idx in enumerate(selected_indices):
        iso = float(isolated_sasa[local_idx])
        complex_val = float(full_complex_sasa[atom_idx])
        buried = float(iso - complex_val)
        rows.append(
            {
                "atom_label": atom_triplet_label(atom_array, atom_idx),
                "isolated_sasa": iso,
                "complex_sasa": complex_val,
                "buried_sasa": buried,
                "fraction_buried": float(buried / iso) if iso > 0 else float("nan"),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
def extract_min_interface_pae(summary_confidences):
    matrix = summary_confidences.get("chain_pair_pae_min") or []
    values = [
        float(value)
        for i, row in enumerate(matrix)
        for j, value in enumerate(row)
        if i != j and value is not None
    ]
    return float(min(values)) if values else np.nan

def compute_candidate_metrics(record):
    reference_complex = record["reference_complex"]
    mobile_complex = record["rf3_output"].atom_array

    binder_backbone_mask_ref = (
        (reference_complex.chain_id == "A")
        & np.isin(reference_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )
    binder_backbone_mask_mobile = (
        (mobile_complex.chain_id == "A")
        & np.isin(mobile_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
    )

    peptide_mask_ref = reference_complex.chain_id == TARGET_CHAIN_ID
    peptide_mask_mobile = mobile_complex.chain_id == TARGET_CHAIN_ID

    peptide_ca_mask_ref = peptide_mask_ref & (reference_complex.atom_name == "CA")
    peptide_ca_mask_mobile = peptide_mask_mobile & (mobile_complex.atom_name == "CA")

    ptr_mask_ref = (
        peptide_mask_ref
        & (reference_complex.res_id == ptm_residue_id)
        & (reference_complex.res_name == "PTR")
    )
    ptr_mask_mobile = (
        peptide_mask_mobile
        & (mobile_complex.res_id == ptm_residue_id)
        & (mobile_complex.res_name == "PTR")
    )

    po4_mask_ref = ptr_mask_ref & np.isin(reference_complex.atom_name, PHOSPHATE_ATOMS)
    po4_mask_mobile = ptr_mask_mobile & np.isin(mobile_complex.atom_name, PHOSPHATE_ATOMS)

    aligned_rf3_complex, binder_transform, binder_alignment_rmsd = align_mobile_on_binder_backbone(
        reference_complex,
        mobile_complex,
        binder_backbone_mask_ref,
        binder_backbone_mask_mobile,
    )

    binder_ref_idx, binder_mobile_idx = paired_common_indices(
        reference_complex,
        mobile_complex,
        binder_backbone_mask_ref,
        binder_backbone_mask_mobile,
    )

    rmsd_rows = [
        {
            "metric": "binder_backbone_alignment_rmsd",
            "rmsd_angstrom": binder_alignment_rmsd,
            "paired_atoms": len(binder_ref_idx),
        }
    ]

    for metric_name, ref_mask, mobile_mask in [
        ("whole_peptide_ca_rmsd", peptide_ca_mask_ref, peptide_ca_mask_mobile),
        ("whole_peptide_all_atom_rmsd", peptide_mask_ref, peptide_mask_mobile),
        ("ptr_all_atom_rmsd", ptr_mask_ref, ptr_mask_mobile),
        ("po4_only_rmsd", po4_mask_ref, po4_mask_mobile),
    ]:
        metric_rmsd, paired_atoms = rmsd_for_masks(
            reference_complex,
            aligned_rf3_complex,
            ref_mask,
            mobile_mask,
            allow_mismatch=True,
        )
        rmsd_rows.append(
            {
                "metric": metric_name,
                "rmsd_angstrom": metric_rmsd,
                "paired_atoms": paired_atoms,
            }
        )

    rmsd_metrics = pd.DataFrame(rmsd_rows)
    rmsd_lookup = dict(zip(rmsd_metrics["metric"], rmsd_metrics["rmsd_angstrom"]))

    hbond_metrics = compute_phosphosite_hbond_metrics(
        aligned_rf3_complex,
        chain_id=TARGET_CHAIN_ID,
        residue_id=ptm_residue_id,
        res_name="PTR",
    )
    po4_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, po4_mask_mobile)
    ptr_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, ptr_mask_mobile)

    summary = record["summary_confidences"]
    overall_plddt = float(summary.get("overall_plddt", np.nan))
    po4_fraction_buried = po4_sasa_metrics["fraction_buried"]
    phosphosite_hbonds = hbond_metrics["phosphosite_hbonds"]

    metric_row = {
        "rf3_name": record["rf3_name"],
        "design_name": record["design_name"],
        "parent_name": record["parent_name"],
        "rfd3_example_id": record["rfd3_example_id"],
        "rfd3_model_index": record["rfd3_model_index"],
        "mpnn_design_idx": record["mpnn_design_idx"],
        "binder_sequence": record["binder_sequence"],
        "sequence_recovery": record["sequence_recovery"],
        "ligand_interface_sequence_recovery": record["ligand_interface_sequence_recovery"],
        "overall_plddt": overall_plddt,
        "overall_plddt_pct": overall_plddt * 100 if np.isfinite(overall_plddt) else np.nan,
        "overall_pae": float(summary.get("overall_pae", np.nan)),
        "min_pae": extract_min_interface_pae(summary),
        "ptm": float(summary.get("ptm", np.nan)) if summary.get("ptm") is not None else np.nan,
        "iptm": float(summary.get("iptm", np.nan)) if summary.get("iptm") is not None else np.nan,
        "ranking_score": float(summary.get("ranking_score", np.nan)) if summary.get("ranking_score") is not None else np.nan,
        "binder_backbone_alignment_rmsd": rmsd_lookup["binder_backbone_alignment_rmsd"],
        "peptide_ca_rmsd": rmsd_lookup["whole_peptide_ca_rmsd"],
        "peptide_all_atom_rmsd": rmsd_lookup["whole_peptide_all_atom_rmsd"],
        "ptr_all_atom_rmsd": rmsd_lookup["ptr_all_atom_rmsd"],
        "po4_only_rmsd": rmsd_lookup["po4_only_rmsd"],
        "po4_fraction_buried": po4_fraction_buried,
        "ptr_fraction_buried": ptr_sasa_metrics["fraction_buried"],
        "phosphosite_hbonds": phosphosite_hbonds,
        "peptide_ca_pass": rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD,
        "po4_burial_pass": po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL,
        "phosphosite_hbonds_pass": phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS,
        "overall_tutorial_pass": (
            rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD
            and po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL
            and phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS
        ),
    }

    detail = {
        **record,
        "aligned_rf3_complex": aligned_rf3_complex,
        "hbond_metrics": hbond_metrics,
        "po4_sasa_metrics": po4_sasa_metrics,
        "ptr_sasa_metrics": ptr_sasa_metrics,
        "metric_row": metric_row,
    }
    return metric_row, detail

scored_rows = []
scored_details = []
for record in rf3_designs:
    metric_row, detail = compute_candidate_metrics(record)
    scored_rows.append(metric_row)
    scored_details.append(detail)

all_scores_df = pd.DataFrame(scored_rows).sort_values(
    by=["overall_tutorial_pass", "overall_plddt", "min_pae"],
    ascending=[False, False, True],
).reset_index(drop=True)

detail_by_name = {detail["rf3_name"]: detail for detail in scored_details}
scored_details = [detail_by_name[name] for name in all_scores_df["rf3_name"]]


In [ ]:
display(all_scores_df.round(3))
print(f"Scored {len(all_scores_df)} RF3-refolded designs.")


In [ ]:
scatter_fig = px.scatter(
    all_scores_df,
    x="min_pae",
    y="overall_plddt_pct",
    color="overall_tutorial_pass",
    hover_name="rf3_name",
    hover_data=[
        "rfd3_model_index",
        "mpnn_design_idx",
        "overall_pae",
        "ranking_score",
        "sequence_recovery",
        "phosphosite_hbonds",
        "po4_fraction_buried",
    ],
    labels={
        "min_pae": "Minimum interface PAE (A)",
        "overall_plddt_pct": "Overall pLDDT (%)",
        "overall_tutorial_pass": "Workshop pass",
    },
    title="RF3 confidence across all generated complexes",
)
scatter_fig.update_traces(marker={"size": 11, "line": {"width": 1, "color": "white"}})
scatter_fig.update_layout(legend_title_text="Workshop pass")
scatter_fig.show()


## 9. Browse the Generated Structures

Use the buttons to walk candidate-by-candidate. The viewer overlays the LigandMPNN design (gray) with the RF3 refold after binder-backbone alignment (red), so you can page through the generated complexes without rerunning any cells.


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

def make_candidate_overlay(detail, width=780, height=480):
    return make_structure_overlay_view(
        detail["reference_complex"],
        detail["aligned_rf3_complex"],
        ptr_chain=TARGET_CHAIN_ID,
        ptr_resi=ptm_residue_id,
        width=width,
        height=height,
    )

viewer_index = {"value": 0}
prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
status_html = widgets.HTML()
note_html = widgets.HTML(
    "<span style='color:#475569'>Gray = LigandMPNN design, red = RF3 refold after binder-backbone alignment.</span>"
)
viewer_output = widgets.Output()

def render_candidate():
    idx = viewer_index["value"]
    detail = scored_details[idx]
    row = all_scores_df.iloc[idx]
    status_html.value = (
        f"<b>{idx + 1}/{len(scored_details)}</b> &nbsp; <code>{row['rf3_name']}</code> &nbsp; "
        f"pLDDT={row['overall_plddt_pct']:.1f}% &nbsp; min_PAE={row['min_pae']:.2f} A"
    )
    prev_button.disabled = idx == 0
    next_button.disabled = idx == len(scored_details) - 1
    with viewer_output:
        clear_output(wait=True)
        display(make_candidate_overlay(detail))
        display(pd.DataFrame([row]).round(3))

def step(delta):
    viewer_index["value"] = min(
        max(viewer_index["value"] + delta, 0),
        len(scored_details) - 1,
    )
    render_candidate()

prev_button.on_click(lambda _: step(-1))
next_button.on_click(lambda _: step(1))

display(
    widgets.VBox(
        [
            widgets.HBox([prev_button, next_button, status_html]),
            note_html,
            viewer_output,
        ]
    )
)
render_candidate()


## 10. Export the Multi-design Results

This writes the aggregate score table plus the RFD3, LigandMPNN, and RF3 structures for the whole sweep into a dedicated output directory.


In [ ]:
EXPORT_DIR = WORK_DIR / "mult_results"
RFD3_EXPORT_DIR = EXPORT_DIR / "rfd3"
MPNN_EXPORT_DIR = EXPORT_DIR / "mpnn"
RF3_EXPORT_DIR = EXPORT_DIR / "rf3"
for path in [EXPORT_DIR, RFD3_EXPORT_DIR, MPNN_EXPORT_DIR, RF3_EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

scores_path = EXPORT_DIR / f"{EXAMPLE_NAME}_all_scores.csv"
all_scores_df.to_csv(scores_path, index=False)

for detail in scored_details:
    rfd3_base = RFD3_EXPORT_DIR / detail["design_name"]
    mpnn_base = MPNN_EXPORT_DIR / detail["design_name"]
    rf3_base = RF3_EXPORT_DIR / detail["rf3_name"]
    to_cif_file(
        detail["rfd3_complex"],
        rfd3_base,
        file_type="cif",
        include_entity_poly=False,
    )
    detail["mpnn_output"].write_structure(base_path=mpnn_base)
    to_cif_file(
        detail["rf3_output"].atom_array,
        rf3_base,
        file_type="cif",
        include_entity_poly=False,
    )

print("Saved multi-design results:")
print(f" - score table: {scores_path}")
print(f" - RFD3 structures: {RFD3_EXPORT_DIR}")
print(f" - LigandMPNN structures: {MPNN_EXPORT_DIR}")
print(f" - RF3 structures: {RF3_EXPORT_DIR}")


## What To Try Next

1. Increase `RFD3_N_BATCHES` or `MPNN_SEQUENCES_PER_BACKBONE` if you want a wider sweep.
2. Sort `all_scores_df` by different columns such as `ranking_score`, `min_pae`, or `po4_fraction_buried`.
3. Use the scatter plot and the previous/next browser together to compare low-PAE, high-pLDDT candidates by eye.
4. Save the strongest binders from `mult_results` for deeper follow-up or downstream pipeline runs.
